docker-compose.yml server (Embedding & Reranker)

```yaml
services:
  embedding-service:
    image: vllm/vllm-openai:latest
    container_name: vllm-qwen3-embedding
    network_mode: host
    runtime: nvidia
    # environment:
    #   - HUGGING_FACE_HUB_TOKEN=${HUGGING_FACE_HUB_TOKEN}  # Optional: Your HF token
    volumes:
      - ./models:/models  # Cache models
    ipc: host  # Shared memory for PyTorch/vLLM

    # --enforce-eager  Optional: For debugging; disable for better perf
    command: >
      --model Qwen/Qwen3-Embedding-0.6B
      --task embed
      --dtype float16
      --max-model-len 8192
      --port 8000
      --enforce-eager
      --gpu-memory-utilization 0.45
    # restart: unless-stopped
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: all
              capabilities: [gpu]

  reranker-service:
    image: vllm/vllm-openai:latest
    container_name: vllm-qwen3-reranker
    network_mode: host
    runtime: nvidia
    # environment:
    #   - HUGGING_FACE_HUB_TOKEN=${HUGGING_FACE_HUB_TOKEN}  # Optional: Your HF token
    volumes:
      - ./models:/models  # Cache models
    ipc: host  # Shared memory for PyTorch/vLLM
    command: >
      --model Qwen/Qwen3-Reranker-0.6B
      --task score
      --dtype float16
      --max-model-len 8192
      --port 8001
      --hf_overrides '{"architectures": ["Qwen3ForSequenceClassification"],"classifier_from_token": ["no", "yes"],"is_original_qwen3_reranker": true}'
      --enforce-eager
      --gpu-memory-utilization 0.45
    # restart: unless-stopped
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: all
              capabilities: [gpu]
```

## [Qwen3-0.6B-Embedding](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)

### Input & Expected output

Input:
```json
[
    "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:What is the capital of China?",
    "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:Explain gravity",
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun."
]
```

Expected output:

```python
[[0.7645568251609802, 0.14142508804798126], [0.13549736142158508, 0.5999549627304077]]
```

In [36]:
import json
import torch

### Langchain OpenAI Embedding Compatible (localhost)

**Bad result!**

In [37]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-0.6B",
    base_url="http://localhost:8000/v1",
    api_key="text",
    # dimensions=1024
)

In [38]:
embeded_docs = embeddings.embed_documents([
"Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:What is the capital of China?",
"Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:Explain gravity",
"The capital of China is Beijing.",
"Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun."
])
embeded_docs = torch.tensor(embeded_docs)
embeded_docs


tensor([[ 0.0075, -0.0630, -0.0175,  ...,  0.0029, -0.0001,  0.0051],
        [ 0.0177,  0.0011, -0.0105,  ..., -0.0167,  0.0029,  0.0191],
        [ 0.0169, -0.0821, -0.0139,  ..., -0.0078,  0.0047,  0.0083],
        [-0.0132, -0.0448, -0.0203,  ..., -0.0386,  0.0135, -0.0149]])

In [39]:

scores = (embeded_docs[:2] @ embeded_docs[2:].T)
print(scores.tolist())

[[0.7125980854034424, 0.6084930896759033], [0.39669960737228394, 0.39812591671943665]]


-------------

### OpenAI Embedding Client (localhost)

**Acceptable!**

In [40]:
from openai import OpenAI

# Modify OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    # defaults to os.environ.get("OPENAI_API_KEY")
    api_key=openai_api_key,
    base_url=openai_api_base,
)

models = client.models.list()
model = models.data[0].id

responses = client.embeddings.create(
    input=[
        "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:What is the capital of China?",
        "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:Explain gravity",
        "The capital of China is Beijing.",
        "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun."
    ],
    model=model,
)

for data in responses.data:
    print(data.embedding)  # list of float of len 1024

[-0.05084228515625, -0.0291748046875, -3.3974647521972656e-05, -0.0260467529296875, -0.036102294921875, -0.0516357421875, 0.050994873046875, 0.0217742919921875, -0.052276611328125, 0.0408935546875, 0.052001953125, 0.0341796875, 0.072021484375, -0.0006070137023925781, -0.030517578125, -0.0056304931640625, 0.00229644775390625, 0.09686279296875, -0.0232696533203125, -0.0124664306640625, 0.03857421875, -0.0166473388671875, 0.046173095703125, -0.06976318359375, 0.08819580078125, 0.0274505615234375, -0.0955810546875, 0.0268096923828125, 0.005367279052734375, 0.00800323486328125, 0.097900390625, 0.0247802734375, 0.00042891502380371094, 0.0243377685546875, 0.02593994140625, -0.003185272216796875, 0.0799560546875, 0.0162353515625, -0.0023174285888671875, -0.00908660888671875, -0.03839111328125, -0.006061553955078125, -0.08929443359375, 0.01378631591796875, -0.0069122314453125, 0.029296875, -0.0143280029296875, -0.009521484375, -0.005306243896484375, -0.0233917236328125, -0.0215606689453125, 0.0

In [41]:
embeded_docs = torch.tensor([data.embedding for data in responses.data])
embeded_docs

tensor([[-5.0842e-02, -2.9175e-02, -3.3975e-05,  ...,  7.4280e-02,
          3.6072e-02, -1.1742e-02],
        [-1.1055e-02, -3.4515e-02, -1.1110e-03,  ..., -2.6520e-02,
          2.7409e-03, -1.3704e-03],
        [-4.7150e-02, -2.0798e-02,  3.6736e-03,  ...,  5.6152e-02,
          7.0679e-02, -1.7105e-02],
        [-5.3009e-02, -1.5106e-02, -1.2484e-03,  ...,  3.6983e-03,
         -2.0523e-02,  1.9638e-02]])

In [42]:
scores = (embeded_docs[:2] @ embeded_docs[2:].T)
print(scores.tolist())

[[0.7643852233886719, 0.1411387324333191], [0.1356509029865265, 0.5996948480606079]]


-------------

### NOVITA OpenAI Embedding Compatible (API)

**Acceptable!**

In [49]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
# load env
NOVITA_BASE_URL=os.getenv("NOVITA_BASE_URL", None)
NOVITA_API_KEY=os.getenv("NOVITA_API_KEY", None)
NOVITA_MODEL_NAME=os.getenv("NOVITA_MODEL_NAME", None)

client = OpenAI(
    base_url=NOVITA_BASE_URL,
    api_key=NOVITA_API_KEY
)

responses = client.embeddings.create(
    input=[
        "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:What is the capital of China?",
        "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:Explain gravity",
        "The capital of China is Beijing.",
        "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun."
    ],
    model=NOVITA_MODEL_NAME,
)

for data in responses.data:
    print(data.embedding)  # list of float of len 1024

[-0.051225584000349045, -0.027583006769418716, -0.000441245996626094, -0.02626952901482582, -0.03572656214237213, -0.0522763654589653, 0.04991210624575615, 0.021278319880366325, -0.051750972867012024, 0.0401923805475235, 0.050962887704372406, 0.03296826034784317, 0.07250390201807022, -0.000989211956039071, -0.030735349282622337, -0.006206176243722439, 0.0012067565694451332, 0.09614647924900055, -0.022197753190994263, -0.012872070074081421, 0.0388789027929306, -0.015958739444613457, 0.04597167670726776, -0.06935156136751175, 0.08826562017202377, 0.027188964188098907, -0.09562108665704727, 0.028371091932058334, 0.005253905896097422, 0.007651000749319792, 0.09772264957427979, 0.024824704974889755, 0.0003878860152326524, 0.024430662393569946, 0.025744140148162842, -0.0037269894964993, 0.080384761095047, 0.015564696863293648, -0.001773193245753646, -0.009128661826252937, -0.038353513926267624, -0.00574645958840847, -0.08826562017202377, 0.013922850601375103, -0.006731566973030567, 0.0294218

In [50]:
embeded_docs = torch.tensor([data.embedding for data in responses.data])
embeded_docs

tensor([[-0.0512, -0.0276, -0.0004,  ...,  0.0746,  0.0360, -0.0124],
        [-0.0106, -0.0336, -0.0011,  ..., -0.0266,  0.0028, -0.0020],
        [-0.0475, -0.0201,  0.0036,  ...,  0.0562,  0.0704, -0.0173],
        [-0.0528, -0.0150, -0.0015,  ...,  0.0034, -0.0212,  0.0203]])

In [51]:
scores = (embeded_docs[:2] @ embeded_docs[2:].T)
print(scores.tolist())

[[0.7662655711174011, 0.1438504159450531], [0.1359432339668274, 0.6011570692062378]]


-------------

### Graphiti OpenAI Embedding Compatible (localhost)

**Acceptable!**

In [61]:
from graphiti_core.embedder.openai import OpenAIEmbedder, OpenAIEmbedderConfig

embedder=OpenAIEmbedder(
    config=OpenAIEmbedderConfig(
        base_url="http://localhost:8000/v1",
        api_key="dummy_text",
        embedding_model="Qwen/Qwen3-Embedding-0.6B",
    )
)

In [65]:
responses = await embedder.client.embeddings.create(
    input=[
        "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:What is the capital of China?",
        "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:Explain gravity",
        "The capital of China is Beijing.",
        "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun."
    ],
    model="Qwen/Qwen3-Embedding-0.6B"
)

for data in responses.data:
    print(data.embedding)  # list of float of len 1024

[-0.05084228515625, -0.0291748046875, -3.3974647521972656e-05, -0.0260467529296875, -0.036102294921875, -0.0516357421875, 0.050994873046875, 0.0217742919921875, -0.052276611328125, 0.0408935546875, 0.052001953125, 0.0341796875, 0.072021484375, -0.0006070137023925781, -0.030517578125, -0.0056304931640625, 0.00229644775390625, 0.09686279296875, -0.0232696533203125, -0.0124664306640625, 0.03857421875, -0.0166473388671875, 0.046173095703125, -0.06976318359375, 0.08819580078125, 0.0274505615234375, -0.0955810546875, 0.0268096923828125, 0.005367279052734375, 0.00800323486328125, 0.097900390625, 0.0247802734375, 0.00042891502380371094, 0.0243377685546875, 0.02593994140625, -0.003185272216796875, 0.0799560546875, 0.0162353515625, -0.0023174285888671875, -0.00908660888671875, -0.03839111328125, -0.006061553955078125, -0.08929443359375, 0.01378631591796875, -0.0069122314453125, 0.029296875, -0.0143280029296875, -0.009521484375, -0.005306243896484375, -0.0233917236328125, -0.0215606689453125, 0.0

In [66]:
embeded_docs = torch.tensor([data.embedding for data in responses.data])
embeded_docs

tensor([[-5.0842e-02, -2.9175e-02, -3.3975e-05,  ...,  7.4280e-02,
          3.6072e-02, -1.1742e-02],
        [-1.1055e-02, -3.4515e-02, -1.1110e-03,  ..., -2.6520e-02,
          2.7409e-03, -1.3704e-03],
        [-4.7150e-02, -2.0798e-02,  3.6736e-03,  ...,  5.6152e-02,
          7.0679e-02, -1.7105e-02],
        [-5.3009e-02, -1.5106e-02, -1.2484e-03,  ...,  3.6983e-03,
         -2.0523e-02,  1.9638e-02]])

In [67]:
scores = (embeded_docs[:2] @ embeded_docs[2:].T)
print(scores.tolist())

[[0.7643852233886719, 0.1411387324333191], [0.1356509029865265, 0.5996948480606079]]


## [Qwen3-0.6B-Reranker]()

### Input & Expected output
[Reference](https://github.com/vllm-project/vllm/pull/19260)

/rerank
```bash
curl http://127.0.0.1:8000/rerank \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{
    "query": "ping",
    "documents": ["pong"],
    "model": "Qwen/Qwen3-Reranker-0.6B"
  }'
```

expected output
```json
{"id":"rerank-43ddc0f96f174ae4a5eef07d51a8defd","model":"Qwen/Qwen3-Reranker-0.6B","usage":{"total_tokens":2},"results":[{"index":0,"document":{"text":"pong"},"relevance_score":0.0673828125}]}
```

### Cohere Rerank Client (vllm)

**Acceptable!**

In [163]:
import cohere

query = "What is the capital of France?"
print(f"Query:\n{query}")

# cohere v1 client
co = cohere.Client(base_url="http://localhost:8001", api_key="sk-fake-key")

# or the v2
co2 = cohere.ClientV2("sk-fake-key", base_url="http://localhost:8001")

Query:
What is the capital of France?


#### Example

In [164]:
rerank_v1_result = co.rerank(
    model="Qwen/Qwen3-Reranker-0.6B",
    query=query,
    documents=[
        "The capital of France is Paris", "Reranking is fun!",
        "The capital of Brazil is Brasilia.",
        "Paris là thủ đô của Pháp",
        "Brasilia là thủ đô của Brazil",
        "vLLM is an open-source framework for fast AI serving"
    ])


print("rerank v1")
print("\n".join([ f"({res.relevance_score:.4f}): {res.document.text}" for res in rerank_v1_result.results]))

###

v2_rerank_result = co2.rerank(
    model="Qwen/Qwen3-Reranker-0.6B",
    query=query,
    documents=[
        "The capital of France is Paris", "Reranking is fun!",
        "The capital of Brazil is Brasilia.",
        "Paris là thủ đô của Pháp",
        "Brasilia là thủ đô của Brazil",
        "vLLM is an open-source framework for fast AI serving"
    ])
print("rerank v2")
print("\n".join([ f"({res.relevance_score:.4f}): {res.document["text"]}" for res in v2_rerank_result.results]))

rerank v1
(0.8930): The capital of Brazil is Brasilia.
(0.8128): Paris là thủ đô của Pháp
(0.5150): Brasilia là thủ đô của Brazil
(0.4233): The capital of France is Paris
(0.1673): Reranking is fun!
(0.0757): vLLM is an open-source framework for fast AI serving
rerank v2
(0.8930): The capital of Brazil is Brasilia.
(0.8128): Paris là thủ đô của Pháp
(0.5150): Brasilia là thủ đô của Brazil
(0.4233): The capital of France is Paris
(0.1673): Reranking is fun!
(0.0757): vLLM is an open-source framework for fast AI serving


#### Input Output check

In [165]:
rerank_v1_result = co.rerank(
    model="Qwen/Qwen3-Reranker-0.6B",
    query="ping",
    documents=[
        "pong"
    ])
print("rerank v1")
print("\n".join([ f"({res.relevance_score:.4f}): {res.document.text}" for res in rerank_v1_result.results]))

###

v2_rerank_result = co2.rerank(
    model="Qwen/Qwen3-Reranker-0.6B",
    query="ping",
    documents=[
        "pong"
    ]
)
print("rerank v2")
print("\n".join([ f"({res.relevance_score:.4f}): {res.document["text"]}" for res in v2_rerank_result.results]))

rerank v1
(0.0662): pong
rerank v2
(0.0662): pong


### 

### Jinaai Rerank Client (vllm)

**Acceptable!**

#### Example

In [166]:
import json
import requests

url = "http://localhost:8001/rerank"

headers = {"accept": "application/json", "Content-Type": "application/json"}

data = {
    "model":
    "Qwen/Qwen3-Reranker-0.6B",
    "query": "What is the capital of France?",
    "documents": [
        "The capital of France is Paris", "Reranking is fun!",
        "The capital of Brazil is Brasilia.",
        "Paris là thủ đô của Pháp",
        "Brasilia là thủ đô của Brazil",
        "vLLM is an open-source framework for fast AI serving"
    ]
}
response = requests.post(url, headers=headers, json=data)

# Check the response
if response.status_code == 200:
    print("Request successful!")
    print("\n".join([ f'{doc["relevance_score"]:.4f}: {doc["document"]["text"]}' for doc in response.json()["results"]]))
else:
    print(f"Request failed with status code: {response.status_code}")
    print(response.text)

Request successful!
0.8930: The capital of Brazil is Brasilia.
0.8128: Paris là thủ đô của Pháp
0.5150: Brasilia là thủ đô của Brazil
0.4233: The capital of France is Paris
0.1673: Reranking is fun!
0.0757: vLLM is an open-source framework for fast AI serving


### Graphiti Reranker

In [175]:
from graphiti_core.cross_encoder.openai_reranker_client import OpenAIRerankerClient
from graphiti_core.llm_client.config import LLMConfig

cross_encoder=OpenAIRerankerClient(
    config=LLMConfig(
        api_key="dummy_text",
        model="Qwen/Qwen3-Reranker-0.6B",
        base_url="http://localhost:8001/v1",
    )
)

...